# Final Workshop — Orders Pipeline

**Databricks Fundamentals | Day 1 | ~45 min**

---

## Scenario

You are a data engineer at a retail company. Your task is to build a **three-stage pipeline** that ingests raw order data, cleans it, and produces a daily revenue summary for the analytics team.

```
[Volume]  orders_batch.json
      │
      ▼
[Task 01]  Bronze Load
      │    Read JSON from Volume → validate schema → write managed Delta table
      │    Output: bronze.lab_orders
      │
      ▼
[Task 02]  Transform
      │    Cast types · calculate net_amount · classify order_tier · filter nulls
      │    Output: bronze.lab_orders_clean
      │
      ▼
[Task 03]  Gold Aggregation
           Group by date · SUM revenue · COUNT orders · AVG order value
           Output: gold.lab_daily_orders
```

Each stage lives in its own notebook (parametrised with Widgets).  
At the end you will assemble all three into a **Databricks Workflow Job** with a scheduled trigger.

---

## Prerequisites

| Requirement | Check |
|-------------|-------|
| Unity Catalog workspace | ✅ provided by trainer |
| `bronze` schema exists in your catalog | ✅ created by `00_setup` |
| `gold` schema exists in your catalog | ✅ created by `00_setup` |
| Dataset path configured | ✅ `DATASET_PATH` from `00_setup` |

---

## Workshop Files

Open each task notebook **in a separate tab** and work through them in order:

| File | What you build |
|------|----------------|
| `tasks/lab_task_01_bronze.ipynb` | Load raw JSON → Delta |
| `tasks/lab_task_02_transform.ipynb` | Apply transformations → clean table |
| `tasks/lab_task_03_gold.ipynb` | Aggregate → daily summary |

> After completing all three notebooks, return here for the **Job creation guide** (Step 4).

---

## Step 1 — Verify Setup


In [ ]:
%run ../../setup/00_setup
print(f"Catalog   : {CATALOG}")
print(f"Bronze    : {BRONZE_SCHEMA}")
print(f"Gold      : {GOLD_SCHEMA}")
print(f"Dataset   : {DATASET_PATH}")

---

## Step 2 — Work through the 3 task notebooks

Open them in order. Each notebook:
- starts with `%run ../../setup/00_setup` (already written)
- has **Widgets** for job parameters (already written — run as-is)
- has `# YOUR CODE HERE` cells you need to complete
- ends with `dbutils.notebook.exit(...)` returning status JSON

**Hints** in the notebooks follow the format:
```
💡 HINT — <function name>:
    <API signature from Databricks documentation>
    <parameter descriptions>
    # NOT the answer — just the API shape
```

---

## Step 3 — Quick Smoke Test

After completing all 3 notebooks, run the cell below to verify all tables exist.


In [ ]:
# Smoke test — verify all three tables were created
tables = [
    f"{CATALOG}.{BRONZE_SCHEMA}.lab_orders",
    f"{CATALOG}.{BRONZE_SCHEMA}.lab_orders_clean",
    f"{CATALOG}.{GOLD_SCHEMA}.lab_daily_orders",
]

all_ok = True
for t in tables:
    try:
        count = spark.table(t).count()
        print(f"  ✅  {t}  →  {count:,} rows")
    except Exception as e:
        print(f"  ❌  {t}  →  {e}")
        all_ok = False

print()
print("All tables ready ✅" if all_ok else "Some tables missing — check task notebooks ❌")


---

## Step 4 — Create the Databricks Workflow Job

### 4.1 New Job

1. Left sidebar → **Workflows** → **Create Job**
2. Job name: `lab_orders_pipeline`

---

### 4.2 Add Task 01 — Bronze Load

| Field | Value |
|-------|-------|
| Task name | `bronze_load` |
| Type | `Notebook` |
| Source | Select `tasks/lab_task_01_bronze` |
| Cluster | *Use existing all-purpose cluster* |
| Key–Value parameters | `source_path` → *(your DATASET_PATH)* |

---

### 4.3 Add Task 02 — Transform

| Field | Value |
|-------|-------|
| Task name | `transform` |
| Depends on | `bronze_load` |
| Type | `Notebook` |
| Source | `tasks/lab_task_02_transform` |
| Cluster | *same as above* |

---

### 4.4 Add Task 03 — Gold

| Field | Value |
|-------|-------|
| Task name | `gold_aggregation` |
| Depends on | `transform` |
| Type | `Notebook` |
| Source | `tasks/lab_task_03_gold` |
| Cluster | *same as above* |

---

### 4.5 Schedule

- Triggers tab → **Add trigger** → **Scheduled**
- Cron: `0 0 6 * * ?`  *(every day at 06:00 UTC)*
- Timezone: `Europe/Warsaw`
- Status: leave as **Paused** for the demo

---

### 4.6 Run and Monitor

1. **Run now** to trigger a manual run
2. Observe the DAG: `bronze_load → transform → gold_aggregation`
3. Click each task → **Output** to see `notebook.exit` JSON
4. If a task fails → **Repair Run** (reruns only failed + downstream tasks)

---

## Checklist

- [ ] `lab_task_01_bronze` — completed and runs without error
- [ ] `lab_task_02_transform` — completed and runs without error
- [ ] `lab_task_03_gold` — completed and runs without error
- [ ] Smoke test cell above shows ✅ for all 3 tables
- [ ] Job created with 3 tasks and correct dependencies
- [ ] Schedule configured (cron + timezone)
- [ ] Manual run succeeded — all tasks green
- [ ] Checked task output JSON in the run logs

---

> ← [M04: Workflows & Automation](../modules/M04_orchestration_jobs.ipynb) | [M00: Intro](../modules/M00_intro.ipynb) →
